<div style="display: flex; background-color: RGB(255,114,0);" >
<h1 style="margin: auto; padding: 30px; ">ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK</h1>
</div>

# OBJECTIF DE CE NOTEBOOK

Bienvenue dans l'outil plébiscité par les analystes de données Jupyter.

Il s'agit d'un outil permettant de mixer et d'alterner code, texte et graphiques.

Cet outil est formidable pour plusieurs raisons:

+ Il permet de tester des lignes de codes au fur et à mesure de votre rédaction, de constater immédiatement le résultat d'une instruction, de la corriger si nécessaire.
+ Il permet aussi de rédiger du texte pour expliquer l'approche suivie ou les résultats d'une analyse et de le mettre en forme grâce à du code html ou plus simple avec **Markdown**
+ Il est possible d'ajouter des graphiques

Pour vous aider dans vos premiers pas à l'usage de Jupyter et de Python, nous avons rédigé ce notebook en vous indiquant les instructions à suivre.

Il vous suffit pour cela de saisir le code Python répondant à l'instruction donnée.

Vous verrez de temps à autre le code Python répondant à une instruction donnée mais cela est fait pour vous aider à comprendre la nature du travail qui vous est demandé.

Et gardez à l'esprit qu'il n'y a pas de solution unique pour résoudre un problème et qu'il y a autant de résolutions de problèmes que de développeurs ;)...



<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

In [ ]:
#Importation de la librairie Pandas
import pandas as pd


In [156]:
#Importation de la librairie plotly express
import plotly.express as px

In [157]:
#Trouver dans Google l'instruction permettant d'afficher toutes les colonnes d'un dataframe
#Saisir dans Google les mots clés "display all columns dataframe Pandas" par exemple.
#Dans les résultats de la recherche, privilégier les solutions provenant de Stack Overflow ou Medium

# https://stackoverflow.com/questions/49188960/how-to-show-all-columns-names-on-a-large-pandas-dataframe
pd.set_option('display.max_columns', None)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargements des fichiers</h3>
</div>

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etapes 1 a 3 (automatisees) - Consolidation des donnees</h2>
</div>

Le chargement, le nettoyage et la fusion des 3 sources (ERP, extraction web, table de liaison) — realises manuellement dans la version initiale du notebook — sont desormais encapsules dans le script `consolidation_donnees.py` (cf. Livrable 4, section Reproductibilite). Le detail des regles de nettoyage appliquees (doublons, `sku` manquants, prix/stocks negatifs, articles hors vente, etc.) reste documente dans le Livrable 1 et dans le rapport d'anomalies genere ci-dessous a chaque execution.

In [ ]:
from consolidation_donnees import consolider_donnees, afficher_rapport

df_merge, rapport = consolider_donnees("erp.xlsx", "web.xlsx", "liaison.xlsx")
afficher_rapport(rapport)
df_merge


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - Analyse univariée des prix</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.1 - Exploration par la visualisation de données</h3>
</div>

In [252]:
#Autre méthode avec plotly express
fig = px.box(df_merge, y="price", title="Distribution des prix des produits", width=600, height=400)
fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2 - Exploration par l'utilisation de méthodes statistiques</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.1 - Identification par le Z-index</h3>
</div>

In [253]:
#Calculer la moyenne du prix
#Calculer l'écart-type du prix

moyenne = df_merge['price'].mean()
ecart_type = df_merge['price'].std()
z_score = (df_merge['price']- moyenne) / ecart_type
print(f"La moyenne du prix est : {moyenne:.2f}, l'écart-type : {ecart_type:.2f}.")

La moyenne du prix est : 32.33, l'écart-type : 27.60.


In [254]:
#Calculer le Z-score (distance en écart-type d'une valeur par rapport à la moyenne)
df_merge['z__score'] = (df_merge['price']- moyenne) / ecart_type

C:\Users\MonCompte\AppData\Local\Temp\ipykernel_26196\286589839.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [255]:
#Quel est le seuil prix dont le z-score est supérieur à 3?
# On peut regarder le premier produit à partir duquel la valeur de z-score dépasse 3
df_merge[['price', 'z__score']][df_merge['z__score'] > 3].sort_values(by='z__score', ascending=True)

,price,z__score
302,116.4,3.046286
406,121.0,3.212975
630,121.0,3.212975
419,122.0,3.249211
153,124.8,3.350674
392,135.0,3.720288
226,137.0,3.792762
212,157.0,4.517496
210,175.0,5.169756
448,176.0,5.205993


In [256]:
# Ou regarder la valeur de prix pour un z-score = 3
seuil = 3 * ecart_type + moyenne
print(f"le seuil pour un z-score à 3 est : {seuil:.2f}.")

le seuil pour un z-score à 3 est : 115.12.


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.2 - Identification par l'intervalle interquartile</h3>
</div>

In [257]:
#Utilisation de la fonction "describe" de Pandas pour l'étude des mesures de dispersion
df_merge.describe()

,product_id,total_sales,stock_quantity,price,purchase_price,z__score
count,714.000000,714.000000,714.000000,714.000000,714.000000,7.140000e+02
mean,5032.557423,8.054622,23.445378,32.333683,16.904006,-4.727000e-17
std,790.510878,4.161344,22.219243,27.596332,14.827384,1.000000e+00
min,3847.000000,0.000000,0.000000,5.200000,2.740000,-9.832351e-01
25%,4280.250000,5.000000,9.000000,14.062500,7.240000,-6.620874e-01
50%,4796.000000,8.000000,20.000000,23.450000,12.305000,-3.219154e-01
75%,5710.500000,11.000000,30.000000,42.075000,22.030000,3.529932e-01
max,7338.000000,36.000000,145.000000,225.000000,137.810000,6.981591e+00


In [258]:
#Définir un seuil pour les articles "outliers" en prix
# On calcule Q1 (25%) et Q3 (75%)
q1 = df_merge['price'].quantile(0.25)
q3 = df_merge['price'].quantile(0.75)

# On calcule l'IQR (l'écart entre les deux)
iqr = q3 - q1

seuil_haut = q3 + 1.5 * iqr

print(f"Le seuil de prix pour les articles outliers est de {seuil_haut:.2f}€.")

Le seuil de prix pour les articles outliers est de 84.09€.


In [259]:
#Définir le nombre d'articles et la proportion de l'ensemble du catalogue "outliers"

#Nombre d'outliers (ceux qui dépassent le seuil)
nb_outliers = len(df_merge[df_merge['price'] > seuil_haut])

#Calcul de la proportion (en pourcentage)
proportion_outliers = (nb_outliers / len(df_merge)) * 100

print(f"Nombre d'outliers détectés : {nb_outliers}")
print(f"Proportion des outliers : {proportion_outliers:.2f} %")

Nombre d'outliers détectés : 31
Proportion des outliers : 4.34 %


In [260]:
#Selon vous, ces outliers sont-ils justifiés ? Comment le démontrer si cela est possible ?
# Je soupconne que ce sont des spiritueux plus chers ou des vins de meilleurs qualité, pour cela il faut aller les regarder
outliers_ventes = df_merge[df_merge['price'] > seuil_haut].sort_values('price', ascending=False)
outliers_ventes

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score
598,4352,Champagne,Champagne Egly-Ouriet Grand Cru Millésimé 2008,11.0,0,225.0,137.81,6.981591
689,5001,Vin,David Duband Charmes-Chambertin Grand Cru 2014,2.0,18,217.5,116.87,6.709816
319,5892,Champagne,Coteaux Champenois Egly-Ouriet Ambonnay Rouge ...,6.0,98,191.3,116.06,5.760415
448,4402,Cognac,Cognac Frapin VIP XO,3.0,11,176.0,78.25,5.205993
210,5767,Vin,Camille Giroud Clos de Vougeot 2016,4.0,12,175.0,90.42,5.169756
212,4406,Cognac,Cognac Frapin Château de Fontpinot 1989 20 Ans...,4.0,12,157.0,69.08,4.517496
226,4904,Vin,Domaine Des Croix Corton Charlemagne Grand Cru...,3.0,9,137.0,67.95,3.792762
392,6126,Champagne,Champagne Gosset Célébris Vintage 2007,5.0,138,135.0,80.33,3.720288
153,5612,Vin,Domaine Weinbach Gewurztraminer Grand Cru Furs...,1.0,19,124.8,66.41,3.350674
419,5917,Whisky,Wemyss Malts Single Cask Scotch Whisky Choc 'n...,3.0,12,122.0,54.24,3.249211


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 5 - Analyse univariée du CA, des quantités vendues, des stocks et de la marge ainsi qu'une analyse multivariée  </h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.1 - Analyse des ventes en CA</h3>
</div>

In [261]:
##############################
# Calculer le CA du site web #
##############################

#Créer une colonne calculant le CA par article
df_merge['CA'] = df_merge['price'] * df_merge['total_sales']
#Calculer la somme de la colonne "ca_par_article"
somme_ca = df_merge['CA'].sum()
#Ce résultat correspond au chiffre d'affaire du site web
print(f"Le chiffre d'affaires total généré par le site web est de : {somme_ca:.2f} €")

Le chiffre d'affaires total généré par le site web est de : 143680.10 €


C:\Users\MonCompte\AppData\Local\Temp\ipykernel_26196\3411840286.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [262]:
###############################
# Palmarès des articles en CA #
###############################

#Effectuer le tri dans l'ordre décroissant du CA du dataset df_merge
df_merge = df_merge.sort_values(by='CA', ascending=False)

In [263]:
#Réinitialiser l'index du dataset par un reset_index
df_merge = df_merge.reset_index(drop=True)

In [264]:
#Afficher les 20 premiers articles en CA
df_topCA = df_merge.head(20)
df_topCA

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA
0,4352,Champagne,Champagne Egly-Ouriet Grand Cru Millésimé 2008,11.0,0,225.0,137.81,6.981591,2475.0
1,5892,Champagne,Coteaux Champenois Egly-Ouriet Ambonnay Rouge ...,6.0,98,191.3,116.06,5.760415,1147.8
2,4353,Champagne,Champagne Egly-Ouriet Grand Cru Brut Rosé,14.0,127,79.5,45.91,1.709152,1113.0
3,5826,Vin,Agnès Levet Côte Rôtie Améthyste 2017,20.0,34,41.2,21.71,0.321286,824.0
4,6212,Vin,Domaine des Comtes Lafon Volnay 1er Cru Santen...,7.0,16,115.0,59.42,2.995554,805.0
5,5026,Champagne,Champagne Agrapart &amp; Fils Minéral Extra Br...,9.0,101,86.8,50.13,1.973680,781.2
6,5008,Vin,Domaine des Comtes Lafon Volnay 1er Cru Santen...,7.0,12,105.0,56.42,2.633187,735.0
7,5767,Vin,Camille Giroud Clos de Vougeot 2016,4.0,12,175.0,90.42,5.169756,700.0
8,6126,Champagne,Champagne Gosset Célébris Vintage 2007,5.0,138,135.0,80.33,3.720288,675.0
9,5025,Champagne,Champagne Agrapart &amp; Fils L'Avizoise Extra...,6.0,136,112.0,68.60,2.886844,672.0


In [265]:
#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(df_topCA, 
             x='CA', 
             y='post_title', 
             color = 'product_type',
             width=1500,
             height=2000,
             title="Top 20 des articles par Chiffre d'Affaires",
             labels={'CA': 'Chiffre d\'Affaires (€)', 'post_title': 'Nom du produit'})

fig.update_layout(yaxis={'categoryorder':'total ascending'}, 
                  height=800)

fig.show()

In [266]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part du CA de la ligne dans le dataset
df_merge['%CA'] = 100 * (df_merge['CA'] / df_merge['CA'].sum())
df_merge

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA,%CA
0,4352,Champagne,Champagne Egly-Ouriet Grand Cru Millésimé 2008,11.0,0,225.0,137.81,6.981591,2475.0,1.722577
1,5892,Champagne,Coteaux Champenois Egly-Ouriet Ambonnay Rouge ...,6.0,98,191.3,116.06,5.760415,1147.8,0.798858
2,4353,Champagne,Champagne Egly-Ouriet Grand Cru Brut Rosé,14.0,127,79.5,45.91,1.709152,1113.0,0.774638
3,5826,Vin,Agnès Levet Côte Rôtie Améthyste 2017,20.0,34,41.2,21.71,0.321286,824.0,0.573496
4,6212,Vin,Domaine des Comtes Lafon Volnay 1er Cru Santen...,7.0,16,115.0,59.42,2.995554,805.0,0.560272
...,...,...,...,...,...,...,...,...,...,...
709,4043,Vin,Pierre Gaillard Côte Rôtie Esprit de Blonde 2017,0.0,0,60.0,29.45,1.002536,0.0,0.000000
710,6038,Vin,I Fabbri Chianti Classico Gran Selezione 2015,0.0,0,48.5,25.31,0.585814,0.0,0.000000
711,4047,Vin,Pierre Gaillard Côtes-du-Rhône Blanc Les Gendr...,0.0,0,18.3,9.93,-0.508534,0.0,0.000000
712,4606,Vin,Catherine et Claude Maréchal Volnay 2017,0.0,0,50.1,24.59,0.643793,0.0,0.000000


In [267]:
#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_merge['%CA_cumule'] = df_merge['%CA'].cumsum()
df_merge

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA,%CA,%CA_cumule
0,4352,Champagne,Champagne Egly-Ouriet Grand Cru Millésimé 2008,11.0,0,225.0,137.81,6.981591,2475.0,1.722577,1.722577
1,5892,Champagne,Coteaux Champenois Egly-Ouriet Ambonnay Rouge ...,6.0,98,191.3,116.06,5.760415,1147.8,0.798858,2.521435
2,4353,Champagne,Champagne Egly-Ouriet Grand Cru Brut Rosé,14.0,127,79.5,45.91,1.709152,1113.0,0.774638,3.296072
3,5826,Vin,Agnès Levet Côte Rôtie Améthyste 2017,20.0,34,41.2,21.71,0.321286,824.0,0.573496,3.869569
4,6212,Vin,Domaine des Comtes Lafon Volnay 1er Cru Santen...,7.0,16,115.0,59.42,2.995554,805.0,0.560272,4.429841
...,...,...,...,...,...,...,...,...,...,...,...
709,4043,Vin,Pierre Gaillard Côte Rôtie Esprit de Blonde 2017,0.0,0,60.0,29.45,1.002536,0.0,0.000000,100.000000
710,6038,Vin,I Fabbri Chianti Classico Gran Selezione 2015,0.0,0,48.5,25.31,0.585814,0.0,0.000000,100.000000
711,4047,Vin,Pierre Gaillard Côtes-du-Rhône Blanc Les Gendr...,0.0,0,18.3,9.93,-0.508534,0.0,0.000000,100.000000
712,4606,Vin,Catherine et Claude Maréchal Volnay 2017,0.0,0,50.1,24.59,0.643793,0.0,0.000000,100.000000


In [268]:
#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% du CA
len(df_merge[df_merge['%CA_cumule'] <= 80])

434

In [269]:
#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print(f"La proportion d'articles représentant 80% du CA est de {100 * len(df_merge[df_merge['%CA_cumule'] <= 80]) / len(df_merge['%CA_cumule']):.2f} %.")

La proportion d'articles représentant 80% du CA est de 60.78 %.


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.2 - Analyse des ventes en quantité</h3>
</div>

In [270]:
#####################################
# Palmarès des articles en quantité #
#####################################

#Effectuer le tri dans l'ordre décroissant de quantités vendues du dataset df_merge
df_merge = df_merge.sort_values(by='total_sales', ascending=False)
df_merge['total_sales']

60     36.0
142    27.0
55     24.0
124    22.0
12     22.0
       ... 
709     0.0
710     0.0
711     0.0
712     0.0
713     0.0
Name: total_sales, Length: 714, dtype: float64

In [271]:
#Réinitialiser l'index du dataset par un reset_index
df_merge = df_merge.reset_index(drop=True)

In [272]:
#Afficher les 20 premiers articles en quantité
df_top_qty = df_merge.head(20)

In [273]:
df_merge['total_sales'].sum()

np.float64(5751.0)

In [274]:
#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(df_top_qty, 
             x='total_sales', 
             y='post_title', 
             color = 'product_type',
             hover_data="post_title",
             width=1500,
             height=2000,
             title="Top 20 des articles par quantités vendues",
             labels={'total_sales': 'Quantités', 'post_title': 'Nom du produit'})

fig.update_layout(yaxis={'categoryorder':'total ascending'}, 
                  height=800)

fig.show()

In [275]:
#############################
# Calculer le 20 / 80 en quantité #
#############################

#Créer une colonne calculant la part en quantité de la ligne dans le dataset
df_merge['%qty'] = 100 * (df_merge['total_sales'] / df_merge['total_sales'].sum())
df_merge


,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA,%CA,%CA_cumule,%qty
0,4867,Vin,Château De La Selve IGP Coteaux de l'Ardèche M...,36.0,121,9.9,4.86,-0.812923,356.4,0.248051,22.799330,0.625978
1,4203,Vin,Mas Laval IGP Pays d'Hérault Les Pampres Blanc...,27.0,74,9.9,5.01,-0.812923,267.3,0.186038,40.239045,0.469484
2,4275,Vin,I Fabbri Chianti Classico Lamole 2017,24.0,62,14.9,7.78,-0.631739,357.6,0.248886,21.557543,0.417319
3,4726,Vin,François Baur Pinot Noir Schlittweg 2017,22.0,0,12.7,6.82,-0.711460,279.4,0.194460,36.807324,0.382542
4,4647,Vin,Bernard Baudry Chinon Rouge La Croix Boissée 2017,22.0,45,28.5,14.14,-0.138920,627.0,0.436386,8.224243,0.382542
...,...,...,...,...,...,...,...,...,...,...,...,...
709,4043,Vin,Pierre Gaillard Côte Rôtie Esprit de Blonde 2017,0.0,0,60.0,29.45,1.002536,0.0,0.000000,100.000000,0.000000
710,6038,Vin,I Fabbri Chianti Classico Gran Selezione 2015,0.0,0,48.5,25.31,0.585814,0.0,0.000000,100.000000,0.000000
711,4047,Vin,Pierre Gaillard Côtes-du-Rhône Blanc Les Gendr...,0.0,0,18.3,9.93,-0.508534,0.0,0.000000,100.000000,0.000000
712,4606,Vin,Catherine et Claude Maréchal Volnay 2017,0.0,0,50.1,24.59,0.643793,0.0,0.000000,100.000000,0.000000


In [276]:
#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_merge['%qty_cumule'] = df_merge['%qty'].cumsum()
df_merge

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA,%CA,%CA_cumule,%qty,%qty_cumule
0,4867,Vin,Château De La Selve IGP Coteaux de l'Ardèche M...,36.0,121,9.9,4.86,-0.812923,356.4,0.248051,22.799330,0.625978,0.625978
1,4203,Vin,Mas Laval IGP Pays d'Hérault Les Pampres Blanc...,27.0,74,9.9,5.01,-0.812923,267.3,0.186038,40.239045,0.469484,1.095462
2,4275,Vin,I Fabbri Chianti Classico Lamole 2017,24.0,62,14.9,7.78,-0.631739,357.6,0.248886,21.557543,0.417319,1.512780
3,4726,Vin,François Baur Pinot Noir Schlittweg 2017,22.0,0,12.7,6.82,-0.711460,279.4,0.194460,36.807324,0.382542,1.895323
4,4647,Vin,Bernard Baudry Chinon Rouge La Croix Boissée 2017,22.0,45,28.5,14.14,-0.138920,627.0,0.436386,8.224243,0.382542,2.277865
...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,4043,Vin,Pierre Gaillard Côte Rôtie Esprit de Blonde 2017,0.0,0,60.0,29.45,1.002536,0.0,0.000000,100.000000,0.000000,100.000000
710,6038,Vin,I Fabbri Chianti Classico Gran Selezione 2015,0.0,0,48.5,25.31,0.585814,0.0,0.000000,100.000000,0.000000,100.000000
711,4047,Vin,Pierre Gaillard Côtes-du-Rhône Blanc Les Gendr...,0.0,0,18.3,9.93,-0.508534,0.0,0.000000,100.000000,0.000000,100.000000
712,4606,Vin,Catherine et Claude Maréchal Volnay 2017,0.0,0,50.1,24.59,0.643793,0.0,0.000000,100.000000,0.000000,100.000000


In [277]:
#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% des ventes en quantité
len(df_merge[df_merge['%qty_cumule'] <= 80])

433

In [278]:
#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print(f"La proportion d'articles représentant 80% du CA est de {100 * len(df_merge[df_merge['%qty_cumule'] <= 80]) / len(df_merge['%qty_cumule']):.2f} %.")

La proportion d'articles représentant 80% du CA est de 60.64 %.


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.3 - Analyse des stocks</h3>
</div>

In [279]:
######################################
# Calculer le nombre de mois de stock #
######################################

#Import de numpy 
import numpy as np
#Création de la colonne Rotation de stock
df_merge['rotation_stock'] = df_merge['stock_quantity'] / df_merge['total_sales']
df_merge['rotation_stock'].describe()

count    692.000000
mean            inf
std             NaN
min        0.000000
25%        1.750000
50%        2.408333
75%        3.100000
max             inf
Name: rotation_stock, dtype: float64

In [280]:
#Remplacement des "inf" par 0
df_merge['rotation_stock'] = df_merge['rotation_stock'].replace(np.inf, 0)
df_merge['rotation_stock']= df_merge['rotation_stock'].fillna(0)

In [281]:
#Effectuer le tri dans l'ordre décroissant du nombre de mois de stock dans le dataset df_merge
df_merge = df_merge.sort_values(by='rotation_stock', ascending=False).reset_index(drop=True)
df_merge['rotation_stock']

0      31.250000
1      27.600000
2      27.000000
3      25.000000
4      23.666667
         ...    
709     0.000000
710     0.000000
711     0.000000
712     0.000000
713     0.000000
Name: rotation_stock, Length: 714, dtype: float64

In [282]:
#Graphique en barre du flop 20 des produits qui ont le plus de mois de stock
flop_20 = df_merge.head(20)

fig = px.bar(flop_20, 
             x='rotation_stock', 
             y='post_title', 
             color='product_type',
             hover_data="post_title",
             width=1500,
             height=2000,
             title="Flop 20 des produits qui ont le plus de mois de stocks",
             labels={'rotation_stock': 'Mois de stocks', 'post_title': 'Nom du produit'})

fig.update_layout(yaxis={'categoryorder':'total ascending'}, 
                  height=800)

fig.show()

In [283]:
####################################
# Valorisation des stocks en euros #
####################################

#Création de la colonne Valorisation des stocks en euros
df_merge['valorisation_euros'] = df_merge['stock_quantity'] * df_merge['price']

In [284]:
df_merge['valorisation_euros']

0       6625.0
1      18630.0
2       4179.6
3       7375.0
4       2662.5
        ...   
709        0.0
710        0.0
711        0.0
712        0.0
713        0.0
Name: valorisation_euros, Length: 714, dtype: float64

In [285]:
#Calculer la somme de la colonne "Valorisation_stock_euros"
print(f"La valeur totale du stock est de {df_merge['valorisation_euros'].sum():,.2f} €.")

La valeur totale du stock est de 494,637.90 €.


In [286]:
##############################################
# Valorisation du nombre de produits en stock #
##############################################

#Calculer la somme de la colonne stock quantity
print(f"La nombre de produits en stock est de {df_merge['stock_quantity'].sum()} bouteilles.")

La nombre de produits en stock est de 16740 bouteilles.


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.4 - Analyse du taux de marge</h3>
</div>

In [287]:
############################
# Analyse du taux de marge #
############################

#Création de la colonne Prix HT
tva = 0.20 
df_merge['prix_ht'] = df_merge['price'] / (1 + tva)

[Source](https://entreprendre.service-public.gouv.fr/vosdroits/F32101) TVA sur les boissons

In [288]:
#Création de la colonne Taux de marge
df_merge['marge'] = df_merge['prix_ht'] - df_merge['purchase_price']

In [289]:
df_merge['taux_marge'] = (df_merge['marge']/df_merge['prix_ht']) * 100

In [290]:
#Afficher le prix minimum de la colonne "taux_marge"
print(f"Le taux de marge minimum est de {df_merge['taux_marge'].min():.2f}%.")

Le taux de marge minimum est de -634.99%.


In [291]:
#Afficher le prix maximum de la colonne "taux_marge"
print(f"Le taux de marge maximum est de {df_merge['taux_marge'].max():.2f}%.")

Le taux de marge maximum est de 47.76%.


In [292]:
#Affichage de la ligne avec un taux de marge inférieur à 0
df_marge_neg = df_merge[df_merge['taux_marge'] < 0]
df_marge_neg

,product_id,product_type,post_title,total_sales,stock_quantity,price,purchase_price,z__score,CA,%CA,%CA_cumule,%qty,%qty_cumule,rotation_stock,valorisation_euros,prix_ht,marge,taux_marge
673,4355,Champagne,Champagne Egly-Ouriet Grand Cru Blanc de Noirs,0.0,97,12.65,77.48,-0.713272,0.0,0.0,100.0,0.0,100.0,0.0,1227.05,10.541667,-66.938333,-634.988142


In [293]:
#probablement une virgule mal placée, le vrai prix doit être 126.5€

In [294]:
#Création d'un dataframe avec les taux positifs
df_benef = df_merge[df_merge['taux_marge'] > 0]

In [295]:
#Afficher le prix minimum de la colonne "taux_marge".
print(f"Le taux de marge minimum est de {df_benef['taux_marge'].min():.2f}%.")

Le taux de marge minimum est de 22.78%.


In [296]:
#Afficher le prix maximum de la colonne "taux_marge"
print(f"Le taux de marge maximum est de {df_benef['taux_marge'].max():.2f}%.")

Le taux de marge maximum est de 47.76%.


In [297]:
df_benef['benef_net'] = (df_benef['taux_marge'] * df_benef['CA']) /100

C:\Users\MonCompte\AppData\Local\Temp\ipykernel_26196\2587008206.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [298]:
#Création d'un dataframe avec le taux de marge moyen par type de produit
df_marge_par_type = df_benef.groupby('product_type')['taux_marge'].mean().reset_index()

In [299]:
df_marge_par_type.rename(columns={'taux_marge': 'taux_marge_moyenne'}, inplace=True)
df_marge_par_type= df_marge_par_type.sort_values(by='taux_marge_moyenne', ascending=False)
df_marge_par_type

,product_type,taux_marge_moyenne
1,Cognac,45.067614
5,Whisky,44.918865
2,Gin,42.800000
4,Vin,38.012262
0,Champagne,28.488539
3,Huile d'olive,24.998198


In [300]:
#Affichage dans un graphique du taux de marge par type de produit

fig = px.bar(df_marge_par_type, 
             x='product_type', 
             y='taux_marge_moyenne', 
             color='product_type',
             hover_data=["product_type"],
             width=1000,
             height=500,
             title="Taux de marge moyen par type de produit",
             labels={'marge_moyenne': 'Marge_moyenne', 'product_type': 'Type de produit'})

fig.update_layout(yaxis={'categoryorder':'total ascending'}, 
                 height=600)

fig.show()

In [301]:
#création d'un dataset avec la somme de bénéfices par type de produit
df_somme_benef_par_type = df_benef.groupby('product_type')['benef_net'].sum().reset_index()

In [302]:
# affichage en diagramme à barres 

fig = px.bar(df_somme_benef_par_type, 
             x='product_type', 
             y='benef_net', 
             color='product_type',
             hover_data=["product_type"],
             text='benef_net',
             width=1000,
             height=500,
             title="Somme des bénéfices par type de produit",
             labels={'benef_net': 'Bénéfices moyen', 'product_type': 'Type de produit'})

fig.update_traces(texttemplate='%{text:.2f} €', textposition='outside')

fig.update_layout(yaxis={'categoryorder':'total ascending'}, 
                 height=600)

fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.5 - Analyse des corrélations entre les variables stock, sales et price</h3>
</div>

In [ ]:
############################
# Analyse des correlations #
############################

# Creation d'une heatmap de correlation avec les variables stock, sales et price (version Plotly)

corr = df_merge[['stock_quantity', 'total_sales', 'price']].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Matrice de correlation"
)
fig.update_layout(width=500, height=450)
fig.show()


In [304]:
corr

,stock_quantity,total_sales,price
stock_quantity,1.000000,0.438930,-0.106654
total_sales,0.438930,1.000000,-0.516258
price,-0.106654,-0.516258,1.000000


In [ ]:
# Version "demi-heatmap" avec Plotly : le triangle superieur est masque (valeurs mises a NaN)

mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)

fig = px.imshow(
    corr_masked,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Matrice de correlation (demi)"
)
fig.update_layout(width=500, height=450)
fig.show()


In [ ]:
# Idem en ajoutant la variable marge

corr = df_merge[['stock_quantity', 'total_sales', 'price', 'marge']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)

fig = px.imshow(
    corr_masked,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Matrice de correlation (demi)"
)
fig.update_layout(width=500, height=450)
fig.show()


In [307]:
#Que peut-on conclure des corrélations ?

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.6 - Mise à disposition de la nouvelle table sur un fichier Excel</h3>
</div>

In [308]:
#Mettre le dataset df_merge sur un fichier Excel
#Cette étape peut être utile pour partager le résultat du dataset obtenu avec les équipes.  
df_merge.to_excel("Analyse_Bottleneck_Finale.xlsx", index=False)

print("Le fichier a été exporté avec succès !")

Le fichier a été exporté avec succès !
